[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/07_optimization/notebooks/02_algoritmos_y_codigo.ipynb)

# Notebook 2: Algoritmos y Código

En este notebook:
1. Implementamos descenso de gradiente desde cero
2. Exploramos el efecto del learning rate
3. Resolvemos problemas con `scipy.optimize`
4. Optimización con restricciones (Lagrange + scipy)
5. Programación lineal
6. Programación entera (MIP)
7. Simulated Annealing (SA)
8. Algoritmos Genéticos (GA)
9. Comparación de todos los métodos

In [ ]:
# Instalar dependencias (ejecutar en Colab)
# !pip install numpy matplotlib scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, minimize_scalar, linprog

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

COLORS = {"blue": "#2E86AB", "red": "#E94F37", "green": "#27AE60",
          "orange": "#F39C12", "purple": "#8E44AD", "gray": "#7F8C8D"}

---
## 1. Descenso de gradiente desde cero

**Teoría:** [7.3 Descenso de gradiente](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#descenso-de-gradiente-sin-restricciones)

La regla es simple:

$$x_{k+1} = x_k - \alpha \nabla f(x_k)$$

- $\nabla f$: dirección de mayor crecimiento
- $-\nabla f$: dirección "cuesta abajo"
- $\alpha$: tamaño del paso (learning rate)

In [ ]:
def gradient_descent(f, grad_f, x0, lr=0.01, n_steps=100):
    """Descenso de gradiente básico.

    Args:
        f: función objetivo f(x) -> float
        grad_f: gradiente grad_f(x) -> array
        x0: punto inicial (array)
        lr: learning rate
        n_steps: número de iteraciones

    Returns:
        trajectory: array (n_steps+1, n) con todos los puntos visitados
    """
    x = x0.copy().astype(float)
    trajectory = [x.copy()]
    for _ in range(n_steps):
        x = x - lr * grad_f(x)
        trajectory.append(x.copy())
    return np.array(trajectory)

In [ ]:
# Probemos con una cuadrática simple: f(x,y) = 3x^2 + y^2
f_quad = lambda x: 3 * x[0]**2 + x[1]**2
grad_quad = lambda x: np.array([6 * x[0], 2 * x[1]])

traj = gradient_descent(f_quad, grad_quad, np.array([3.0, 3.0]), lr=0.08, n_steps=25)

print(f"Inicio:  ({traj[0, 0]:.3f}, {traj[0, 1]:.3f}), f = {f_quad(traj[0]):.3f}")
print(f"Final:   ({traj[-1, 0]:.3f}, {traj[-1, 1]:.3f}), f = {f_quad(traj[-1]):.3f}")
print(f"Óptimo:  (0, 0), f = 0")

---
## 2. Visualizar la trayectoria de GD

In [ ]:
def plot_gd_contour(f_2d, trajectory, x_range=(-4, 4), y_range=(-4, 4),
                    title="Descenso de gradiente", n_levels=20):
    """Grafica contornos de f con la trayectoria de GD."""
    xg = np.linspace(*x_range, 300)
    yg = np.linspace(*y_range, 300)
    X, Y = np.meshgrid(xg, yg)
    Z = f_2d(X, Y)

    fig, ax = plt.subplots(figsize=(8, 7))
    cs = ax.contour(X, Y, Z, levels=n_levels, cmap="viridis", alpha=0.7)
    ax.clabel(cs, inline=True, fontsize=8)

    ax.plot(trajectory[:, 0], trajectory[:, 1], "o-", color=COLORS["red"],
            markersize=4, linewidth=1.5, label="Trayectoria GD")
    ax.scatter(*trajectory[0], color=COLORS["orange"], s=100, zorder=5,
              edgecolors="black", label="Inicio")
    ax.scatter(*trajectory[-1], color=COLORS["green"], s=100, zorder=5,
              edgecolors="black", label="Final")

    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    plt.show()

plot_gd_contour(
    lambda X, Y: 3*X**2 + Y**2,
    traj,
    title=r"GD en $f(x,y) = 3x^2 + y^2$, lr=0.08"
)

---
## 3. Ejercicio: Explorar learning rates

**Teoría:** [7.3 El learning rate importa mucho](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#el-learning-rate-importa-mucho)

Prueba 3 learning rates diferentes y observa qué pasa.

In [ ]:
learning_rates = [0.01, 0.08, 0.30]  # <-- CHANGE THIS: prueba valores como 0.001, 0.5, etc.

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

xg = np.linspace(-4, 4, 300)
yg = np.linspace(-4, 4, 300)
X, Y = np.meshgrid(xg, yg)
Z = 3*X**2 + Y**2

for ax, lr in zip(axes, learning_rates):
    traj_lr = gradient_descent(f_quad, grad_quad, np.array([3.0, 3.0]), lr=lr, n_steps=30)

    ax.contour(X, Y, Z, levels=20, cmap="viridis", alpha=0.5)
    ax.plot(traj_lr[:, 0], traj_lr[:, 1], "o-", color=COLORS["red"],
            markersize=3, linewidth=1)
    ax.scatter(*traj_lr[0], color=COLORS["orange"], s=80, zorder=5, edgecolors="black")
    ax.scatter(*traj_lr[-1], color=COLORS["green"], s=80, zorder=5, edgecolors="black")
    ax.set_title(f"lr = {lr}")
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)

fig.suptitle("Efecto del learning rate", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

### Reflexión: Learning rate

- El learning rate **crítico** para $f(x,y) = 3x^2 + y^2$ es $\alpha_{\text{crit}} = 2/L_{\max}$ donde $L_{\max} = 6$ (eigenvalor más grande del Hessiano). Así que $\alpha_{\text{crit}} = 1/3 \approx 0.333$.
- Prueba `lr=0.33` — converge lento pero estable. Ahora prueba `lr=0.34` — ¿qué pasa?
- Este es un resultado teórico: GD converge si y solo si $\alpha < 2/L_{\max}$ para cuadráticas.

---
## SGD vs Batch GD

**Teoría:** [7.3 Descenso de gradiente estocástico](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#descenso-de-gradiente-estocástico-sgd)

En ML, la función objetivo es una suma sobre datos: $f(w) = \frac{1}{N}\sum_i f_i(w)$.
SGD estima el gradiente con un **mini-batch** aleatorio en lugar de usar todos los datos.
Veamos la diferencia en un problema de regresión lineal sintética.

In [ ]:
# Datos sintéticos: y = 2*x + 1 + ruido
np.random.seed(42)
N = 200
X_data = np.random.randn(N, 1)
y_data = 2 * X_data[:, 0] + 1 + 0.5 * np.random.randn(N)

# Loss: MSE = (1/N) * sum((y - Xw)^2), w = [slope, intercept]
X_aug = np.hstack([X_data, np.ones((N, 1))])  # add bias column

def mse_loss(w):
    return np.mean((y_data - X_aug @ w)**2)

def mse_grad_full(w):
    return -2/N * X_aug.T @ (y_data - X_aug @ w)

def mse_grad_sgd(w, batch_size=32):
    idx = np.random.choice(N, batch_size, replace=False)
    X_b, y_b = X_aug[idx], y_data[idx]
    return -2/len(idx) * X_b.T @ (y_b - X_b @ w)

# Run both
w0 = np.array([0.0, 0.0])
n_steps = 150
lr = 0.05

# Batch GD
losses_gd = []
w_gd = w0.copy()
for _ in range(n_steps):
    losses_gd.append(mse_loss(w_gd))
    w_gd = w_gd - lr * mse_grad_full(w_gd)

# SGD (batch_size=32)
losses_sgd = []
w_sgd = w0.copy()
for _ in range(n_steps):
    losses_sgd.append(mse_loss(w_sgd))
    w_sgd = w_sgd - lr * mse_grad_sgd(w_sgd, batch_size=32)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses_gd, linewidth=2, color=COLORS["blue"], label="Batch GD (N=200)")
ax.plot(losses_sgd, linewidth=1.5, color=COLORS["red"], alpha=0.8, label="SGD (batch=32)")
ax.set_xlabel("Iteración")
ax.set_ylabel("MSE Loss")
ax.set_title("SGD vs Batch GD en regresión lineal")
ax.legend()
ax.set_yscale("log")
plt.show()

print(f"Batch GD final: w = [{w_gd[0]:.3f}, {w_gd[1]:.3f}]")
print(f"SGD final:      w = [{w_sgd[0]:.3f}, {w_sgd[1]:.3f}]")
print(f"Verdadero:      w = [2.000, 1.000]")

---
## 4. Rosenbrock: cuando GD sufre

**Teoría:** [7.3 Métodos de segundo orden](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#métodos-de-segundo-orden--intuición) | [7.4 Patrón 2](https://www.sonder.art/ia_p26/07_optimization/04_ejemplos_python/#patrón-2-minimización-multidimensional--minimize)

La función de Rosenbrock es famosa por su "valle banana" donde GD avanza muy lento:

$$f(x,y) = (1-x)^2 + 100(y - x^2)^2$$

Mínimo global en $(1, 1)$ con $f = 0$.

In [ ]:
rosenbrock = lambda x: (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2
grad_rosen = lambda x: np.array([
    -2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2),
    200 * (x[1] - x[0]**2),
])

traj_rosen = gradient_descent(rosenbrock, grad_rosen,
                               np.array([-1.5, 2.0]), lr=0.001, n_steps=5000)

print(f"Después de 5000 pasos: ({traj_rosen[-1, 0]:.4f}, {traj_rosen[-1, 1]:.4f})")
print(f"Distancia al óptimo (1,1): {np.linalg.norm(traj_rosen[-1] - [1, 1]):.4f}")

In [ ]:
# Visualizar
rosen_2d = lambda X, Y: (1 - X)**2 + 100 * (Y - X**2)**2

xg = np.linspace(-2, 2, 400)
yg = np.linspace(-1, 3, 400)
X, Y = np.meshgrid(xg, yg)
Z = rosen_2d(X, Y)

fig, ax = plt.subplots(figsize=(9, 7))
cs = ax.contour(X, Y, np.log1p(Z), levels=30, cmap="inferno", alpha=0.7)

# Subsample trajectory
step = max(1, len(traj_rosen) // 200)
traj_sub = traj_rosen[::step]
ax.plot(traj_sub[:, 0], traj_sub[:, 1], "-", color=COLORS["blue"],
        linewidth=1, alpha=0.7)
ax.scatter(*traj_rosen[0], color=COLORS["orange"], s=100, zorder=5, edgecolors="black", label="Inicio")
ax.scatter(*traj_rosen[-1], color=COLORS["green"], s=100, zorder=5, edgecolors="black", label="Final")
ax.scatter(1, 1, color=COLORS["red"], s=120, zorder=5, marker="*", edgecolors="black", label="Óptimo (1,1)")

ax.set_title("GD en Rosenbrock (5000 pasos, lr=0.001)")
ax.legend()
plt.show()

### Reflexión: Rosenbrock y segundo orden

- GD usa 5000 pasos y no llega al óptimo. L-BFGS-B llega en ~50 evaluaciones. ¿Por qué?
- L-BFGS-B usa **curvatura aproximada** (información de segundo orden) para tomar pasos más inteligentes.
- En el "valle banana", la curvatura es muy diferente en la dirección del valle vs perpendicular a él. GD no lo sabe; L-BFGS-B sí.

---
## 5. scipy.optimize: minimización 1D

**Teoría:** [7.4 Patrón 1](https://www.sonder.art/ia_p26/07_optimization/04_ejemplos_python/#patrón-1-minimización-1d--minimize_scalar)

Para problemas 1D, `minimize_scalar` es directo.

In [ ]:
f_1d = lambda x: (x - 3)**2 + 2 * np.sin(5 * x)

result = minimize_scalar(f_1d, bounds=(0, 6), method="bounded")
print(f"Mínimo encontrado: x = {result.x:.4f}")
print(f"f(x) = {result.fun:.4f}")

# Visualizar
x = np.linspace(-1, 7, 800)
fig, ax = plt.subplots()
ax.plot(x, f_1d(x), linewidth=2, color=COLORS["blue"],
        label=r"$f(x) = (x-3)^2 + 2\sin(5x)$")
ax.scatter(result.x, result.fun, color=COLORS["red"], s=120, zorder=5,
           edgecolors="black")
ax.annotate(f"Mínimo: ({result.x:.2f}, {result.fun:.2f})",
            xy=(result.x, result.fun), xytext=(result.x + 1, result.fun + 3),
            arrowprops=dict(arrowstyle="->"), fontsize=11, color=COLORS["red"])
ax.set_title("Minimización 1D con scipy")
ax.legend()
plt.show()

---
## 6. scipy.optimize: minimización 2D (Rosenbrock)

**Teoría:** [7.4 Patrón 2](https://www.sonder.art/ia_p26/07_optimization/04_ejemplos_python/#patrón-2-minimización-multidimensional--minimize)

Comparemos nuestro GD casero con scipy (que usa L-BFGS-B por defecto).

In [ ]:
result_scipy = minimize(rosenbrock, x0=np.array([-1.5, 2.0]), method="L-BFGS-B")

print(f"scipy L-BFGS-B:")
print(f"  Solución: ({result_scipy.x[0]:.6f}, {result_scipy.x[1]:.6f})")
print(f"  f(x) = {result_scipy.fun:.10f}")
print(f"  Evaluaciones de f: {result_scipy.nfev}")
print(f"\nNuestro GD (5000 pasos):")
print(f"  Solución: ({traj_rosen[-1, 0]:.6f}, {traj_rosen[-1, 1]:.6f})")
print(f"  f(x) = {rosenbrock(traj_rosen[-1]):.10f}")

---
## 7. Optimización con restricciones: Lagrange

**Teoría:** [7.3 Multiplicadores de Lagrange](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#multiplicadores-de-lagrange-restricciones-de-igualdad)

**Problema:** $\min x^2 + y^2$ sujeto a $x + y = 1$

**Por Lagrange:**

$\mathcal{L}(x,y,\lambda) = x^2 + y^2 + \lambda(x + y - 1)$

Resolviendo:
- $\partial_x: 2x + \lambda = 0$
- $\partial_y: 2y + \lambda = 0$
- $\partial_\lambda: x + y - 1 = 0$

→ $x = y = 1/2$, $f^* = 1/2$

In [ ]:
# Verificar con scipy
f_constrained = lambda x: x[0]**2 + x[1]**2
constraint = {"type": "eq", "fun": lambda x: x[0] + x[1] - 1}

result_lag = minimize(f_constrained, x0=[0.0, 0.0], constraints=constraint)

print(f"Solución analítica: (0.5, 0.5), f = 0.5")
print(f"Solución scipy:     ({result_lag.x[0]:.4f}, {result_lag.x[1]:.4f}), f = {result_lag.fun:.4f}")

In [ ]:
# Visualizar: contornos de f con la restricción
xg = np.linspace(-1, 2, 300)
yg = np.linspace(-1, 2, 300)
X, Y = np.meshgrid(xg, yg)
Z = X**2 + Y**2

fig, ax = plt.subplots(figsize=(7, 7))
cs = ax.contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.7)
ax.clabel(cs, inline=True, fontsize=8)

# Constraint line: x + y = 1
ax.plot(xg, 1 - xg, "--", color=COLORS["red"], linewidth=2, label="x + y = 1")

# Solution
ax.scatter(0.5, 0.5, color=COLORS["red"], s=150, zorder=5, edgecolors="black",
           label="Óptimo (0.5, 0.5)")

ax.set_title(r"$\min\ x^2 + y^2$ sujeto a $x + y = 1$")
ax.legend()
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()

---
## 8. scipy con restricciones (dict)

Puedes pasar restricciones de igualdad y desigualdad como diccionarios.

In [ ]:
# Ejemplo: min x^2 + y^2  s.t.  x + y >= 2, x >= 0, y >= 0
f_ex = lambda x: x[0]**2 + x[1]**2

constraints = [
    {"type": "ineq", "fun": lambda x: x[0] + x[1] - 2},  # x+y >= 2  (ineq means >= 0)
]
bounds = [(0, None), (0, None)]

result_ineq = minimize(f_ex, x0=[2.0, 2.0], constraints=constraints, bounds=bounds)
print(f"Solución: ({result_ineq.x[0]:.4f}, {result_ineq.x[1]:.4f})")
print(f"f(x) = {result_ineq.fun:.4f}")
print(f"x+y = {result_ineq.x[0] + result_ineq.x[1]:.4f} (debe ser >= 2)")

---
## 9. Programación lineal con `linprog`

**Teoría:** [7.3 Método simplex](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#método-simplex-programación-lineal)

Resolvemos el problema de producción:

$$\max\ 5x_1 + 4x_2 \quad \text{s.t.} \quad 6x_1 + 4x_2 \leq 24,\quad x_1 + 2x_2 \leq 6,\quad x_1, x_2 \geq 0$$

`linprog` **minimiza**, así que negamos $c$.

In [ ]:
c = [-5, -4]              # min -5x1 - 4x2 = max 5x1 + 4x2
A_ub = [[6, 4], [1, 2]]
b_ub = [24, 6]
bounds = [(0, None), (0, None)]

result_lp = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

print(f"Solución: x1 = {result_lp.x[0]:.2f}, x2 = {result_lp.x[1]:.2f}")
print(f"Ganancia máxima: {-result_lp.fun:.2f}")

In [ ]:
# Visualizar la región factible + solución
fig, ax = plt.subplots(figsize=(8, 7))

x1 = np.linspace(0, 5, 400)
c1 = (24 - 6*x1) / 4
c2 = (6 - x1) / 2

ax.plot(x1, c1, color=COLORS["blue"], linewidth=2, label=r"$6x_1 + 4x_2 \leq 24$")
ax.plot(x1, c2, color=COLORS["green"], linewidth=2, label=r"$x_1 + 2x_2 \leq 6$")

y_upper = np.minimum(c1, c2)
y_upper = np.maximum(y_upper, 0)
ax.fill_between(x1, 0, y_upper, where=(y_upper >= 0) & (x1 >= 0),
                alpha=0.2, color=COLORS["blue"], label="Región factible")

sol = result_lp.x
ax.scatter(sol[0], sol[1], color=COLORS["red"], s=200, zorder=6,
           edgecolors="black", linewidths=2, marker="*")
ax.annotate(f"Óptimo: ({sol[0]:.1f}, {sol[1]:.1f})\nz = {-result_lp.fun:.0f}",
            xy=(sol[0], sol[1]), xytext=(sol[0]+0.3, sol[1]+0.6),
            arrowprops=dict(arrowstyle="->"), fontsize=11, color=COLORS["red"],
            fontweight="bold")

ax.set_xlim(-0.5, 5)
ax.set_ylim(-0.5, 5)
ax.set_title("Programación lineal: solución con scipy.linprog")
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.legend()
plt.show()

### Reflexión: Programación lineal

- La solución siempre está en un **vértice** del politopo (región factible).
- Cambia `c = [-4, -5]` (intercambia ganancias). ¿La solución salta a otro vértice?
- Cambia `b_ub = [24, 10]` (más recurso B). ¿La región factible crece? ¿Cambia el óptimo?

---
## Diferenciación automática (autodiff)

**Teoría:** [7.4 Autodiff](https://www.sonder.art/ia_p26/07_optimization/04_ejemplos_python/#diferenciación-automática-autodiff)

Hasta ahora calculamos gradientes **a mano**. Autodiff los calcula exactamente y automáticamente — es lo que hace `loss.backward()` en PyTorch.

In [ ]:
# Autodiff con PyTorch — compara con gradiente analítico de Rosenbrock
try:
    import torch

    # Mismo punto que usamos antes
    xy = torch.tensor([-1.5, 2.0], requires_grad=True)
    loss = (1 - xy[0])**2 + 100 * (xy[1] - xy[0]**2)**2
    loss.backward()

    grad_autodiff = xy.grad.numpy()
    grad_manual = grad_rosen(np.array([-1.5, 2.0]))

    print("Gradiente de Rosenbrock en (-1.5, 2.0):")
    print(f"  Autodiff (PyTorch): [{grad_autodiff[0]:.4f}, {grad_autodiff[1]:.4f}]")
    print(f"  Analítico (a mano): [{grad_manual[0]:.4f}, {grad_manual[1]:.4f}]")
    print(f"  ¿Coinciden? {np.allclose(grad_autodiff, grad_manual)}")
    print("\nEsto es lo que hace loss.backward() en cada paso de entrenamiento.")
except ImportError:
    print("PyTorch no está instalado.")
    print("En Colab: !pip install torch")
    print("El gradiente analítico de Rosenbrock en (-1.5, 2.0) es:")
    grad_manual = grad_rosen(np.array([-1.5, 2.0]))
    print(f"  [{grad_manual[0]:.4f}, {grad_manual[1]:.4f}]")

---
## Optimización entera

**Teoría:** [7.2 Optimización entera / discreta](https://www.sonder.art/ia_p26/07_optimization/02_paisaje_y_conceptos/#optimización-entera--discreta)

Cuando las variables deben ser **enteras**, no puedes simplemente redondear la solución continua.
`scipy.optimize.milp` resuelve programas lineales con variables enteras (MIP).

In [ ]:
from scipy.optimize import milp, LinearConstraint, Bounds

# === Knapsack con 6 objetos ===
# Objetos: A(5kg,$10), B(4kg,$8), C(6kg,$11), D(3kg,$7), E(7kg,$14), F(2kg,$3)
names = ["A", "B", "C", "D", "E", "F"]
weights = [5, 4, 6, 3, 7, 2]
values = [10, 8, 11, 7, 14, 3]
capacity = 15

c = [-v for v in values]  # milp minimiza → negamos
constraints = LinearConstraint([weights], ub=capacity)
integrality = [1] * 6
bounds = Bounds(lb=0, ub=1)  # binarias

# Solución MIP (óptima)
result_mip = milp(c, constraints=constraints, integrality=integrality, bounds=bounds)
selected_ip = [names[i] for i in range(6) if result_mip.x[i] > 0.5]
print(f"Solución IP: {selected_ip}")
print(f"  Peso: {sum(w*x for w,x in zip(weights, result_mip.x)):.0f} kg")
print(f"  Valor: {-result_mip.fun:.0f}")

# Relajación continua (sin integrality)
result_relax = milp(c, constraints=constraints, bounds=Bounds(lb=0, ub=1))
print(f"\nRelajación LP: {[f'{names[i]}={result_relax.x[i]:.2f}' for i in range(6)]}")
print(f"  Valor LP: {-result_relax.fun:.2f} (cota superior)")
print(f"  Gap: {-result_relax.fun - (-result_mip.fun):.2f}")

# Visualización: barras de items seleccionados
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: IP solution - selected items
ip_selected = [result_mip.x[i] > 0.5 for i in range(6)]
bar_colors = [COLORS["green"] if s else COLORS["gray"] for s in ip_selected]
axes[0].bar(names, values, color=bar_colors, edgecolor="black")
axes[0].set_title(f"IP: items seleccionados (valor={-result_mip.fun:.0f})")
axes[0].set_ylabel("Valor ($)")
for i, (v, s) in enumerate(zip(values, ip_selected)):
    axes[0].text(i, v + 0.3, f"{'✓' if s else '✗'}\n{weights[i]}kg", ha="center", fontsize=10)

# Panel 2: LP relaxation vs IP
x_pos = np.arange(6)
axes[1].bar(x_pos - 0.2, result_relax.x, 0.35, color=COLORS["blue"], label="LP (continua)", edgecolor="black")
axes[1].bar(x_pos + 0.2, result_mip.x, 0.35, color=COLORS["green"], label="IP (entera)", edgecolor="black")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(names)
axes[1].set_ylabel("$x_i$")
axes[1].set_title("LP relajación vs IP: variables de decisión")
axes[1].legend()
axes[1].set_ylim(0, 1.15)

fig.suptitle("Knapsack: redondear la LP NO da la solución IP óptima", fontsize=13)
fig.tight_layout()
plt.show()

### Reflexión: Programación entera

- La relajación LP da una **cota superior** del valor óptimo. El gap LP-IP mide qué tan lejos está.
- **Branch-and-bound** usa estas cotas LP para podar ramas del árbol de búsqueda.
- El solver HiGHS (que usa `milp` internamente) es open-source y competitivo con solvers comerciales para problemas medianos.
- Para problemas industriales grandes (miles de variables enteras), CPLEX y Gurobi son el estándar.

---
## Simulated Annealing (SA)

**Teoría:** [7.3 Simulated Annealing](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#simulated-annealing-recocido-simulado)

Cuando no tienes gradiente, SA es una opción simple: genera vecinos aleatorios, acepta mejoras siempre, y acepta empeoramientos con probabilidad $p = e^{-\Delta/T}$ que decrece con la temperatura.

Usaremos la **función de Rastrigin** — tiene muchos mínimos locales, con el global en el origen:

$$f(x) = 10n + \sum_{i=1}^n \left[ x_i^2 - 10\cos(2\pi x_i) \right]$$

In [ ]:
# Rastrigin 2D: muchos mínimos locales, global en (0, 0) con f = 0
def rastrigin(x):
    return 10 * len(x) + sum(xi**2 - 10 * np.cos(2 * np.pi * xi) for xi in x)

def rastrigin_2d(X, Y):
    return 20 + X**2 + Y**2 - 10*np.cos(2*np.pi*X) - 10*np.cos(2*np.pi*Y)

# Visualizar la superficie
g = np.linspace(-5.12, 5.12, 300)
X, Y = np.meshgrid(g, g)
Z = rastrigin_2d(X, Y)

fig, ax = plt.subplots(figsize=(8, 7))
cf = ax.contourf(X, Y, Z, levels=30, cmap="viridis")
fig.colorbar(cf, ax=ax, label="f(x, y)")
ax.scatter(0, 0, color="red", s=150, zorder=5, marker="*", edgecolors="black",
           label="Global min (0, 0)")
ax.set_title("Rastrigin 2D: muchos mínimos locales")
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.legend()
plt.show()
print(f"f(0, 0) = {rastrigin([0, 0]):.1f}  (global)")
print(f"f(1, 1) = {rastrigin([1, 1]):.4f}  (local)")

In [ ]:
def simulated_annealing(f, bounds, T0=10.0, alpha=0.995, n_iter=3000, seed=42):
    """Simulated Annealing from scratch."""
    rng = np.random.default_rng(seed)
    dim = len(bounds)
    x = np.array([rng.uniform(lo, hi) for lo, hi in bounds])
    best_x, best_f = x.copy(), f(x)
    T = T0
    history_x, history_T, history_best = [x.copy()], [T], [best_f]

    for _ in range(n_iter):
        x_new = x + rng.normal(0, 0.5, size=dim)
        x_new = np.clip(x_new, [lo for lo, _ in bounds], [hi for _, hi in bounds])
        f_old, f_new = f(x), f(x_new)
        delta = f_new - f_old
        # Metropolis acceptance criterion
        if delta < 0 or rng.random() < np.exp(-delta / max(T, 1e-10)):
            x = x_new
            if f(x) < best_f:
                best_x, best_f = x.copy(), f(x)
        T *= alpha
        history_x.append(x.copy())
        history_T.append(T)
        history_best.append(best_f)

    return best_x, best_f, np.array(history_x), history_T, history_best

# Run SA on Rastrigin
best_x_sa, best_f_sa, hist_x_sa, hist_T_sa, hist_best_sa = simulated_annealing(
    rastrigin, [(-5.12, 5.12)] * 2, T0=10.0, alpha=0.995, n_iter=3000
)
print(f"SA resultado: x = ({best_x_sa[0]:.4f}, {best_x_sa[1]:.4f})")
print(f"SA f(x) = {best_f_sa:.4f}")
print(f"Global:  f(0, 0) = 0")

In [ ]:
# SA visualization: 3 panels
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: trajectory on contours
ax = axes[0]
ax.contourf(X, Y, Z, levels=30, cmap="viridis", alpha=0.8)
ax.plot(hist_x_sa[:, 0], hist_x_sa[:, 1], "-", color="white", linewidth=0.3, alpha=0.4)
n_sa = len(hist_x_sa)
colors_traj = plt.cm.hot(np.linspace(0, 0.8, n_sa))
ax.scatter(hist_x_sa[::50, 0], hist_x_sa[::50, 1], c=colors_traj[::50],
           s=8, zorder=3, edgecolors="none")
ax.scatter(*hist_x_sa[0], color=COLORS["orange"], s=100, zorder=5,
           edgecolors="black", label="Inicio")
ax.scatter(*best_x_sa, color=COLORS["green"], s=120, zorder=5,
           edgecolors="black", marker="*", label=f"Mejor: f={best_f_sa:.2f}")
ax.scatter(0, 0, color="white", s=80, zorder=5, marker="x", linewidths=2)
ax.set_title("Trayectoria SA"); ax.legend(fontsize=8)

# Panel 2: temperature (log)
ax = axes[1]
ax.plot(hist_T_sa, color=COLORS["red"], linewidth=1.5)
ax.set_yscale("log")
ax.set_title("Temperatura T vs iteración")
ax.set_xlabel("Iteración"); ax.set_ylabel("T (log)")
ax.axhline(y=1.0, color=COLORS["gray"], linestyle="--", linewidth=0.8, label="T=1")
ax.legend()

# Panel 3: best-so-far
ax = axes[2]
ax.plot(hist_best_sa, color=COLORS["blue"], linewidth=1.5)
ax.set_title("Mejor f(x) encontrado")
ax.set_xlabel("Iteración"); ax.set_ylabel("f(x*)")
ax.axhline(y=0, color=COLORS["green"], linestyle="--", linewidth=0.8, label="Global (0)")
ax.legend()

fig.suptitle("Simulated Annealing en Rastrigin 2D", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

### Reflexión: Simulated Annealing

- **Temperatura alta** = acepta empeoramientos (exploración). **Temperatura baja** = solo mejoras (explotación).
- Prueba cambiar `T0` (1 vs 100) y `alpha` (0.9 vs 0.999). ¿Qué pasa si enfrías muy rápido?
- Prueba diferentes `x0` — ¿siempre encuentra el global?
- `scipy.optimize.dual_annealing` es una versión mejorada que combina SA con búsqueda local.

In [ ]:
from scipy.optimize import dual_annealing

result_da = dual_annealing(rastrigin, [(-5.12, 5.12)] * 2, seed=42)

print(f"dual_annealing:")
print(f"  x = ({result_da.x[0]:.6f}, {result_da.x[1]:.6f})")
print(f"  f(x) = {result_da.fun:.6f}")
print(f"  Evaluaciones: {result_da.nfev}")
print(f"\nNuestro SA:")
print(f"  x = ({best_x_sa[0]:.6f}, {best_x_sa[1]:.6f})")
print(f"  f(x) = {best_f_sa:.6f}")
print(f"  Evaluaciones: ~{len(hist_x_sa) * 2}")

---
## Algoritmo Genético (GA)

**Teoría:** [7.3 Algoritmos Genéticos](https://www.sonder.art/ia_p26/07_optimization/03_algoritmos/#algoritmos-genéticos-ga)

En lugar de un solo punto (como SA), GA mantiene una **población** de soluciones que evoluciona:
selección, crossover, mutación, reemplazo. Usemos la misma función de Rastrigin para comparar.

In [ ]:
def genetic_algorithm(f, bounds, pop_size=60, n_gen=100, mutation_rate=0.3,
                      mutation_scale=0.5, seed=42):
    """Genetic Algorithm from scratch."""
    rng = np.random.default_rng(seed)
    dim = len(bounds)
    lo = np.array([b[0] for b in bounds])
    hi = np.array([b[1] for b in bounds])

    # Initialize population
    pop = rng.uniform(lo, hi, size=(pop_size, dim))
    fitness = np.array([f(ind) for ind in pop])

    history_best, history_mean = [], []
    snapshots = {}  # gen -> pop copy

    for gen in range(n_gen):
        history_best.append(fitness.min())
        history_mean.append(fitness.mean())
        if gen in (0, n_gen // 3, 2 * n_gen // 3, n_gen - 1):
            snapshots[gen] = pop.copy()

        # Tournament selection
        new_pop = []
        for _ in range(pop_size):
            i, j = rng.choice(pop_size, 2, replace=False)
            winner = pop[i] if fitness[i] < fitness[j] else pop[j]
            new_pop.append(winner.copy())
        new_pop = np.array(new_pop)

        # BLX-alpha crossover
        for k in range(0, pop_size - 1, 2):
            alpha_blx = 0.5
            d = np.abs(new_pop[k] - new_pop[k + 1])
            c_lo = np.minimum(new_pop[k], new_pop[k + 1]) - alpha_blx * d
            c_hi = np.maximum(new_pop[k], new_pop[k + 1]) + alpha_blx * d
            new_pop[k] = rng.uniform(c_lo, c_hi)
            new_pop[k + 1] = rng.uniform(c_lo, c_hi)

        # Gaussian mutation
        mask = rng.random(pop_size) < mutation_rate
        new_pop[mask] += rng.normal(0, mutation_scale, size=(mask.sum(), dim))
        new_pop = np.clip(new_pop, lo, hi)

        # Elitism: keep best from previous generation
        new_fitness = np.array([f(ind) for ind in new_pop])
        best_idx = np.argmin(fitness)
        worst_idx = np.argmax(new_fitness)
        new_pop[worst_idx] = pop[best_idx]
        new_fitness[worst_idx] = fitness[best_idx]

        pop, fitness = new_pop, new_fitness

    return pop, fitness, history_best, history_mean, snapshots

# Run GA on Rastrigin
pop_ga, fit_ga, hist_best_ga, hist_mean_ga, snaps_ga = genetic_algorithm(
    rastrigin, [(-5.12, 5.12)] * 2, pop_size=60, n_gen=100
)
best_ga = pop_ga[np.argmin(fit_ga)]
print(f"GA resultado: x = ({best_ga[0]:.4f}, {best_ga[1]:.4f})")
print(f"GA f(x) = {fit_ga.min():.4f}")
print(f"Global:  f(0, 0) = 0")

In [ ]:
# GA visualization: 3 panels
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: population snapshots
ax = axes[0]
ax.contourf(X, Y, Z, levels=30, cmap="viridis", alpha=0.6)
gen_colors = {0: COLORS["red"], 33: COLORS["orange"],
              66: COLORS["blue"], 99: COLORS["green"]}
for gen_k, snap in snaps_ga.items():
    ax.scatter(snap[:, 0], snap[:, 1], s=15, color=gen_colors[gen_k],
               alpha=0.7, label=f"Gen {gen_k}")
ax.scatter(0, 0, color="white", s=80, zorder=5, marker="x", linewidths=2)
ax.set_title("Población por generación"); ax.legend(fontsize=8)

# Panel 2: best/mean fitness
ax = axes[1]
ax.plot(hist_best_ga, color=COLORS["green"], linewidth=2, label="Mejor")
ax.plot(hist_mean_ga, color=COLORS["blue"], linewidth=1.5, alpha=0.7, label="Media")
ax.set_title("Fitness por generación")
ax.set_xlabel("Generación"); ax.set_ylabel("f(x)")
ax.axhline(y=0, color=COLORS["gray"], linestyle="--", linewidth=0.8)
ax.legend()

# Panel 3: final fitness histogram
ax = axes[2]
ax.hist(fit_ga, bins=20, color=COLORS["purple"], edgecolor="black", alpha=0.8)
ax.axvline(fit_ga.min(), color=COLORS["red"], linewidth=2, linestyle="--",
           label=f"Mejor: {fit_ga.min():.2f}")
ax.set_title("Fitness última generación")
ax.set_xlabel("f(x)"); ax.set_ylabel("Frecuencia")
ax.legend()

fig.suptitle("Algoritmo Genético en Rastrigin 2D", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

### Reflexión: Algoritmo Genético

- La población se **concentra** cerca del óptimo conforme avanzan las generaciones.
- Prueba cambiar `pop_size` (10 vs 200) y `mutation_rate` (0.01 vs 0.8). ¿Más población siempre ayuda?
- **Diversidad**: si la mutación es muy baja, la población converge prematuramente a un local.
- `scipy.optimize.differential_evolution` es una variante evolutiva optimizada.

In [ ]:
from scipy.optimize import differential_evolution

result_de = differential_evolution(rastrigin, [(-5.12, 5.12)] * 2, seed=42)

print(f"differential_evolution:")
print(f"  x = ({result_de.x[0]:.6f}, {result_de.x[1]:.6f})")
print(f"  f(x) = {result_de.fun:.6f}")
print(f"  Evaluaciones: {result_de.nfev}")
print(f"\nNuestro GA:")
print(f"  x = ({best_ga[0]:.6f}, {best_ga[1]:.6f})")
print(f"  f(x) = {fit_ga.min():.6f}")
print(f"  Evaluaciones: ~{60 * 100}")

---
## Comparación: todos los métodos en Rastrigin 2D

Pongamos **todos** los métodos a competir en la misma función. ¿Cuáles encuentran el mínimo global?

In [ ]:
# Run ALL methods on Rastrigin from the same starting point
x0_comp = np.array([3.0, 3.0])
bounds_comp = [(-5.12, 5.12)] * 2

# 1. GD (custom, using numerical gradient of Rastrigin)
def rastrigin_grad(x):
    return np.array([2*xi + 10*2*np.pi*np.sin(2*np.pi*xi) for xi in x])

x_gd = x0_comp.copy()
for _ in range(2000):
    x_gd = x_gd - 0.001 * rastrigin_grad(x_gd)
    x_gd = np.clip(x_gd, -5.12, 5.12)
f_gd = rastrigin(x_gd)

# 2. L-BFGS-B
res_bfgs = minimize(rastrigin, x0_comp, method="L-BFGS-B",
                    bounds=bounds_comp)

# 3. SA (our custom)
best_sa2, f_sa2, _, _, _ = simulated_annealing(rastrigin, bounds_comp)

# 4. GA (our custom)
pop2, fit2, _, _, _ = genetic_algorithm(rastrigin, bounds_comp)
f_ga2 = fit2.min()

# 5. dual_annealing
res_da2 = dual_annealing(rastrigin, bounds_comp, seed=42)

# 6. differential_evolution
res_de2 = differential_evolution(rastrigin, bounds_comp, seed=42)

# Print comparison table
print(f"{'Método':<25} {'f(x)':>10} {'Global?':>10} {'Evals':>10}")
print("-" * 58)
methods = [
    ("GD (custom)", f_gd, f_gd < 1.0, 2000),
    ("L-BFGS-B", res_bfgs.fun, res_bfgs.fun < 1.0, res_bfgs.nfev),
    ("SA (custom)", f_sa2, f_sa2 < 1.0, 6000),
    ("GA (custom)", f_ga2, f_ga2 < 1.0, 6000),
    ("dual_annealing", res_da2.fun, res_da2.fun < 1.0, res_da2.nfev),
    ("diff_evolution", res_de2.fun, res_de2.fun < 1.0, res_de2.nfev),
]
for name, fval, glob, nev in methods:
    mark = "Si" if glob else "No"
    print(f"{name:<25} {fval:>10.4f} {mark:>10} {nev:>10}")

### Reflexión: Comparación de métodos

- Los métodos con **gradiente** (GD, L-BFGS-B) convergen rápido pero se atascan en el **mínimo local más cercano**.
- Las **metaheurísticas** (SA, GA) pueden encontrar el global pero necesitan más evaluaciones.
- `dual_annealing` y `differential_evolution` de scipy son versiones optimizadas que generalmente superan a implementaciones caseras.

**Regla práctica:**
- Si tienes gradiente y el problema es convexo: **L-BFGS-B** o **GD**
- Si no tienes gradiente o hay muchos mínimos locales: **`dual_annealing`** o **`differential_evolution`**
- Si el problema es discreto/combinatorio: **GA** o **SA**

---
## 10. Ejercicio capstone

**Teoría:** [7.4 Ejercicio capstone](https://www.sonder.art/ia_p26/07_optimization/04_ejemplos_python/#ejercicio-capstone-a-mano-y-con-scipy)

**Problema:** Una empresa quiere minimizar costos de transporte:

$$f(x_1, x_2) = 2x_1^2 + 3x_2^2 + x_1 x_2$$

sujeto a $x_1 + x_2 \geq 10$, $x_1, x_2 \geq 0$.

**Tareas:**
1. Resuelve con Lagrange (asumiendo restricción activa)
2. Verifica con scipy
3. Visualiza la solución en contornos

In [ ]:
# 1. Tu solución analítica (escribe los valores)
x1_analitico = 6.25   # <-- CHANGE THIS
x2_analitico = 3.75   # <-- CHANGE THIS

f_capstone = lambda x: 2*x[0]**2 + 3*x[1]**2 + x[0]*x[1]
print(f"Solución analítica: ({x1_analitico}, {x2_analitico})")
print(f"f = {f_capstone([x1_analitico, x2_analitico]):.4f}")

In [ ]:
# 2. Verificar con scipy
constraint_cap = {"type": "ineq", "fun": lambda x: x[0] + x[1] - 10}
bounds_cap = [(0, None), (0, None)]

result_cap = minimize(f_capstone, x0=[5.0, 5.0], constraints=constraint_cap, bounds=bounds_cap)

print(f"scipy: ({result_cap.x[0]:.4f}, {result_cap.x[1]:.4f})")
print(f"f = {result_cap.fun:.4f}")
print(f"x1 + x2 = {result_cap.x[0] + result_cap.x[1]:.4f}")

In [ ]:
# 3. Visualizar
xg = np.linspace(0, 12, 300)
yg = np.linspace(0, 12, 300)
X, Y = np.meshgrid(xg, yg)
Z = 2*X**2 + 3*Y**2 + X*Y

fig, ax = plt.subplots(figsize=(8, 7))
cs = ax.contour(X, Y, Z, levels=25, cmap="viridis", alpha=0.7)
ax.clabel(cs, inline=True, fontsize=8)

# Constraint: x + y = 10
ax.plot(xg, 10 - xg, "--", color=COLORS["red"], linewidth=2, label="x + y = 10")
ax.fill_between(xg, 10 - xg, 12, alpha=0.1, color=COLORS["green"], label="Factible")

# Solution
sol_cap = result_cap.x
ax.scatter(sol_cap[0], sol_cap[1], color=COLORS["red"], s=150, zorder=5,
           edgecolors="black", marker="*", label=f"Óptimo ({sol_cap[0]:.2f}, {sol_cap[1]:.2f})")

ax.set_xlim(0, 12)
ax.set_ylim(0, 12)
ax.set_title(r"$\min\ 2x_1^2 + 3x_2^2 + x_1 x_2$ s.t. $x_1 + x_2 \geq 10$")
ax.legend()
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
plt.show()